# 01 - Data Pull

**MScFE 690 Capstone - Deep Learning for Dynamic Multi-Asset Portfolio Allocation**

This notebook downloads and stores the raw data used by every later stage: daily adjusted prices for the ETF universe (plus the frozen four-name equity sleeve AAPL, JPM, JNJ, XOM committed at M2) and the FRED macro series. It is scripted and idempotent: re-running it refreshes the CSV store under `/data` and rewrites the cleaning log. No manual edits to raw files are allowed; every transformation is recorded here so the report can cite the pipeline rather than a hand-edited file.

Design decisions referenced in the M2 Problem Statement: sample window from 2015-01-01 to the present; adjusted closes for splits and dividends; calendar alignment across tickers before any feature work.

*Path note:* all file locations resolve from `src.config.project_root()`, so this notebook runs identically from the repository root or from `notebooks/`.

In [ ]:
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import pandas as pd
import numpy as np
import yfinance as yf

from src.cleaning import set_seed
from src.config import load_config, project_root

CFG = load_config()
ROOT = project_root()

# Reproducibility pre-commitment: one shared code path pins the python and
# numpy generators from config/config.yaml before any pull or downstream
# numeric work (tensorflow is seeded in notebook 03, the first model stage).
set_seed(int(CFG["seeds"]["numpy"]))

ETF_UNIVERSE = CFG["universe"]["etfs"]            # 10 sector SPDRs + SPY
EQUITY_SLEEVE = CFG["universe"]["equity_sleeve"]  # frozen at M2
MACRO_SERIES = CFG["macro"]["fred_series"]
START = CFG["sample"]["start"]

os.makedirs(ROOT / "data/raw", exist_ok=True)
print("ETF universe:", ETF_UNIVERSE)
print("Equity sleeve (frozen):", EQUITY_SLEEVE)
print("FRED series:", MACRO_SERIES)

## 1. Price download

`auto_adjust=True` returns split- and dividend-adjusted OHLC, which is the series our return calculations require. We keep the adjusted close and record the download date so results are reproducible against the stored snapshot.

In [ ]:
tickers = ETF_UNIVERSE + EQUITY_SLEEVE
raw = yf.download(tickers, start=START, auto_adjust=True, progress=False)

# yf.download returns a MultiIndex frame even for a single ticker; flatten to
# one adjusted-close column per symbol so the CSV schema never changes.
if isinstance(raw.columns, pd.MultiIndex):
    prices = raw["Close"] if "Close" in raw.columns.get_level_values(0) else raw.xs("Close", axis=1, level=-1)
else:
    prices = raw[["Close"]].rename(columns={"Close": tickers[0]})
prices = prices.dropna(how="all").sort_index()
prices.to_csv(ROOT / "data/raw/prices.csv")
print("Downloaded", prices.shape[0], "rows x", prices.shape[1], "tickers")
print("Coverage start:", prices.index.min().date(), "end:", prices.index.max().date())

## 2. FRED macro series

Secondary features from FRED (VIXCLS, DGS10, T10Y2Y, DFF, CPIAUCSL). Availability is confirmed per series at this pull. A missing or renamed FRED series aborts the notebook with a message pointing at `macro.fred_series` in the config - a silent gap would shrink the joined sample later without anyone noticing.

In [ ]:
from pandas_datareader import data as pdr

macro = {}
for series in MACRO_SERIES:
    try:
        macro[series] = pdr.DataReader(series, "fred", start=START)[series]
    except Exception as exc:
        # Loud failure, not a silent gap: a missing macro column would
        # otherwise drop whole rows later and shrink the sample invisibly.
        raise RuntimeError(
            f"FRED series {series!r} unavailable ({exc}). Name a substitute "
            "in config/config.yaml under macro.fred_series before pulling."
        ) from exc
assert macro, "No FRED series could be pulled; aborting instead of storing an empty frame."
macro_df = pd.DataFrame(macro)
macro_df.to_csv(ROOT / "data/raw/macro.csv")
print(macro_df.tail())

## 3. Data-quality gate

The gate reports missing values per ticker and per series. The cleaning policy (forward-fill limits, drop rule, outlier screens) is applied in notebook 02, not here: raw stays raw. A pull that fails the gate aborts with a logged reason instead of passing bad data forward.

In [ ]:
missing = prices.isna().sum()
print("Missing adjusted closes per ticker:")
print(missing[missing > 0] if (missing > 0).any() else "none")
gate_ok = bool((missing <= CFG["cleaning"]["max_missing_days"]).all())
print("DATA GATE:", "PASS" if gate_ok else "FAIL - inspect before continuing")
assert gate_ok, "Data gate failed; fix the pull before running notebook 02."